# SQL for Senior Data Engineers: 100 Interview-Focused Q&A

**Scope**: 100 practical SQL patterns for Senior Data Engineers across ANSI/warehouse dialects. Each item includes a concise **problem**, a reference **SQL** query, and a short **explanation**. Assume ANSI SQL; some items note dialect options (Databricks/Presto/Redshift/BigQuery/Postgres).

---

### 1) Select specific columns and filter rows

**Schema**: table: sales(order_id INT, customer_id INT, amount DECIMAL(12,2), currency STRING, order_date DATE)

**Problem**: Return order_id, customer_id, amount for orders in 2026 with amount > 100 and currency = "USD".

**Reference (SQL)**:
```sql

SELECT order_id, customer_id, amount
FROM sales
WHERE order_date >= DATE '2026-01-01'
  AND order_date <  DATE '2027-01-01'
  AND amount > 100
  AND currency = 'USD';
```
**Explanation**: Basic filtering with half-open date range to enable partition pruning and avoid BETWEEN fencepost errors.

---

### 2) Distinct vs. GROUP BY for de-duplication

**Schema**: table: customers(customer_id INT, email STRING, country STRING)

**Problem**: Return distinct country codes present in customers.

**Reference (SQL)**:
```sql

SELECT DISTINCT country
FROM customers;
```
**Explanation**: DISTINCT is syntactic sugar for GROUP BY when projecting only the distinct columns.

---

### 3) Case-insensitive search

**Schema**: table: products(product_id INT, name STRING)

**Problem**: Find products whose name contains 'lens' (case-insensitive).

**Reference (SQL)**:
```sql

SELECT *
FROM products
WHERE LOWER(name) LIKE '%lens%';
```
**Explanation**: Normalize both sides (LOWER) or use case-insensitive collations where supported.

---

### 4) IN vs. EXISTS

**Schema**: table: orders(order_id INT, customer_id INT); table: vip(custom_id INT)

**Problem**: Select orders placed by VIP customers. Prefer EXISTS for large left side.

**Reference (SQL)**:
```sql

SELECT o.*
FROM orders o
WHERE EXISTS (
  SELECT 1 FROM vip v WHERE v.custom_id = o.customer_id
);
```
**Explanation**: EXISTS can short-circuit and often optimizes better than IN for correlated filtering.

---

### 5) Safe NULL handling

**Schema**: table: visits(id INT, utm_source STRING)

**Problem**: Return rows where utm_source is NULL or empty after trimming.

**Reference (SQL)**:
```sql

SELECT *
FROM visits
WHERE utm_source IS NULL OR TRIM(utm_source) = '';
```
**Explanation**: Use IS NULL for nulls; TRIM/empty-string for blanks—do not conflate the two.

---

### 6) Boolean expression simplification

**Schema**: table: txns(id INT, status STRING)

**Problem**: Return non-terminal statuses: anything not in ('SUCCESS','FAILED').

**Reference (SQL)**:
```sql

SELECT *
FROM txns
WHERE status NOT IN ('SUCCESS','FAILED') OR status IS NULL;
```
**Explanation**: NOT IN with NULL in the list yields UNKNOWN; keep explicit NULL handling.

---

### 7) Top-N without ties (ANSI FETCH FIRST)

**Schema**: table: sales(order_id INT, amount DECIMAL(12,2))

**Problem**: Return top 10 orders by amount.

**Reference (SQL)**:
```sql

SELECT order_id, amount
FROM sales
ORDER BY amount DESC
FETCH FIRST 10 ROWS ONLY;
```
**Explanation**: ANSI syntax; in other dialects use LIMIT 10.

---

### 8) Search across multiple columns

**Schema**: table: users(id INT, email STRING, phone STRING, alt_phone STRING)

**Problem**: Find users whose primary or alternate phone matches a value.

**Reference (SQL)**:
```sql

SELECT *
FROM users
WHERE phone = :p OR alt_phone = :p;
```
**Explanation**: Avoid OR on many columns if possible; consider UNPIVOT/UNION ALL for predicate pushdown in big tables.

---

### 9) Coalesce for survivorship

**Schema**: table: contacts(id INT, mobile STRING, home STRING, work STRING)

**Problem**: Choose best-available phone among mobile/home/work.

**Reference (SQL)**:
```sql

SELECT id, COALESCE(mobile, home, work) AS primary_phone
FROM contacts;
```
**Explanation**: COALESCE returns the first non-NULL expression.

---

### 10) Simple text normalization

**Schema**: table: dim_customer(customer_id INT, name STRING)

**Problem**: Lower-case and trim names on select.

**Reference (SQL)**:
```sql

SELECT customer_id, TRIM(LOWER(name)) AS name_norm
FROM dim_customer;
```
**Explanation**: Normalize at read for idempotent transformations; persist only when needed.

---

### 11) Inner join on natural key

**Schema**: orders(order_id INT, customer_id INT), customers(customer_id INT, country STRING)

**Problem**: Enrich orders with customer country.

**Reference (SQL)**:
```sql

SELECT o.order_id, o.customer_id, c.country
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id;
```
**Explanation**: Equi-join on matching key names; alias for readability.

---

### 12) Left join with default via COALESCE

**Schema**: orders(order_id INT, coupon_code STRING), coupons(code STRING, discount INT)

**Problem**: Attach discount, defaulting to 0 when coupon missing.

**Reference (SQL)**:
```sql

SELECT o.order_id, COALESCE(c.discount, 0) AS discount
FROM orders o
LEFT JOIN coupons c ON c.code = o.coupon_code;
```
**Explanation**: LEFT JOIN preserves left rows; COALESCE provides fallback.

---

### 13) Anti-join (rows in A not in B)

**Schema**: table_a(id INT), table_b(id INT)

**Problem**: Return rows in A whose id is not present in B.

**Reference (SQL)**:
```sql

SELECT a.*
FROM table_a a
LEFT JOIN table_b b ON b.id = a.id
WHERE b.id IS NULL;
```
**Explanation**: ANSI anti-join via LEFT JOIN ... WHERE b.key IS NULL.

---

### 14) Semi-join (existence)

**Schema**: orders(order_id INT, customer_id INT), risky_customers(customer_id INT)

**Problem**: Return orders with risky customers (no right columns).

**Reference (SQL)**:
```sql

SELECT o.*
FROM orders o
WHERE EXISTS (
  SELECT 1 FROM risky_customers r WHERE r.customer_id = o.customer_id
);
```
**Explanation**: Use EXISTS to filter by existence without pulling right-side columns.

---

### 15) UNION ALL vs UNION

**Schema**: table x(a INT), table y(a INT)

**Problem**: Stack two tables; keep duplicates.

**Reference (SQL)**:
```sql

SELECT a FROM x
UNION ALL
SELECT a FROM y;
```
**Explanation**: UNION ALL avoids DISTINCT step; cheaper when duplicates are allowed.

---

### 16) Full outer join for change capture

**Schema**: src(key INT, value STRING), tgt(key INT, value STRING)

**Problem**: Identify inserts/updates/deletes between src and tgt by key.

**Reference (SQL)**:
```sql

SELECT COALESCE(s.key, t.key) AS key,
       s.value AS src_value,
       t.value AS tgt_value,
       CASE
         WHEN s.key IS NOT NULL AND t.key IS NULL THEN 'INSERT'
         WHEN s.key IS NULL AND t.key IS NOT NULL THEN 'DELETE'
         WHEN s.value <> t.value THEN 'UPDATE'
         ELSE 'UNCHANGED'
       END AS change_type
FROM src s
FULL OUTER JOIN tgt t ON t.key = s.key;
```
**Explanation**: FULL OUTER JOIN surfaces all differences for simple CDC audits.

---

### 17) Join on date range (type-2 lookup)

**Schema**: fact(order_id INT, order_date DATE, region_id INT), dim_region(region_id INT, region STRING, eff_start DATE, eff_end DATE)

**Problem**: Attach region name valid on the order_date.

**Reference (SQL)**:
```sql

SELECT f.order_id, d.region
FROM fact f
JOIN dim_region d
  ON d.region_id = f.region_id
 AND f.order_date >= d.eff_start
 AND f.order_date <  d.eff_end;
```
**Explanation**: Range join for SCD2 validity; prefer half-open intervals to avoid overlaps.

---

### 18) Join with composite keys

**Schema**: a(k1 INT, k2 INT, v INT), b(k1 INT, k2 INT, x INT)

**Problem**: Join on (k1,k2).

**Reference (SQL)**:
```sql

SELECT *
FROM a
JOIN b ON a.k1 = b.k1 AND a.k2 = b.k2;
```
**Explanation**: Composite joins require matching all key columns.

---

### 19) Self-join to compare rows

**Schema**: prices(sku STRING, dt DATE, price DECIMAL(10,2))

**Problem**: Find dates when a SKU’s price increased relative to previous day.

**Reference (SQL)**:
```sql

SELECT p1.sku, p1.dt
FROM prices p1
JOIN prices p0
  ON p0.sku = p1.sku AND p0.dt = p1.dt - INTERVAL '1' DAY
WHERE p1.price > p0.price;
```
**Explanation**: Self-join on date offsets to detect changes across rows.

---

### 20) EXCEPT / MINUS for set difference

**Schema**: events_a(e STRING), events_b(e STRING)

**Problem**: Events present in A but not in B.

**Reference (SQL)**:
```sql

SELECT e FROM events_a
EXCEPT
SELECT e FROM events_b;
```
**Explanation**: Use EXCEPT (or MINUS) for set difference on distinct rows.

---

### 21) Basic aggregations

**Schema**: sales(order_id INT, store_id INT, amount DECIMAL(12,2))

**Problem**: Compute total, average, and max sales per store.

**Reference (SQL)**:
```sql

SELECT store_id,
       SUM(amount) AS total_sales,
       AVG(amount) AS avg_sales,
       MAX(amount) AS max_sale
FROM sales
GROUP BY store_id;
```
**Explanation**: Grouped aggregations compute multiple metrics in one pass.

---

### 22) Count distinct efficiently (approximate)

**Schema**: events(customer_id INT)

**Problem**: Approximate unique customers.

**Reference (SQL)**:
```sql

-- Many warehouses provide approx functions (e.g., HLL)
SELECT APPROX_COUNT_DISTINCT(customer_id) AS approx_unique_customers
FROM events;
```
**Explanation**: Approximate NDV (e.g., HyperLogLog) is faster and memory efficient for very large data.

---

### 23) Conditional aggregation with CASE

**Schema**: sales(channel STRING, amount DECIMAL(12,2))

**Problem**: Total revenue split by channel.

**Reference (SQL)**:
```sql

SELECT SUM(amount) AS total,
       SUM(CASE WHEN channel = 'online'  THEN amount ELSE 0 END) AS online,
       SUM(CASE WHEN channel = 'store'   THEN amount ELSE 0 END) AS store,
       SUM(CASE WHEN channel = 'partner' THEN amount ELSE 0 END) AS partner
FROM sales;
```
**Explanation**: CASE inside aggregates computes conditional sums in a single scan.

---

### 24) HAVING vs WHERE

**Schema**: orders(customer_id INT, amount DECIMAL(12,2))

**Problem**: Customers whose total spend > 1000.

**Reference (SQL)**:
```sql

SELECT customer_id, SUM(amount) AS total_spend
FROM orders
GROUP BY customer_id
HAVING SUM(amount) > 1000;
```
**Explanation**: WHERE filters rows before grouping; HAVING filters groups after aggregation.

---

### 25) Group sets: ROLLUP

**Schema**: sales(store_id INT, category STRING, amount DECIMAL(12,2))

**Problem**: Compute totals by (store, category), by store, and grand total.

**Reference (SQL)**:
```sql

SELECT store_id, category, SUM(amount) AS total
FROM sales
GROUP BY ROLLUP (store_id, category);
```
**Explanation**: ROLLUP produces hierarchical subtotals and grand totals; identify with GROUPING() if needed.

---

### 26) CUBE for cross-tab totals

**Schema**: sales(region STRING, channel STRING, amount DECIMAL(12,2))

**Problem**: Totals for all combinations of region and channel.

**Reference (SQL)**:
```sql

SELECT region, channel, SUM(amount) AS total
FROM sales
GROUP BY CUBE (region, channel);
```
**Explanation**: CUBE yields all subtotal combinations; can be heavy—use sparingly.

---

### 27) GROUPING SETS custom totals

**Schema**: sales(region STRING, product STRING, amount DECIMAL(12,2))

**Problem**: Totals by region and by product only (no cross subtotal).

**Reference (SQL)**:
```sql

SELECT region, product, SUM(amount) AS total
FROM sales
GROUP BY GROUPING SETS ((region), (product));
```
**Explanation**: GROUPING SETS allows arbitrary subtotal combinations.

---

### 28) Median with percentile_disc

**Schema**: payments(amount DECIMAL(12,2))

**Problem**: Compute median payment amount.

**Reference (SQL)**:
```sql

SELECT PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY amount) AS median
FROM payments;
```
**Explanation**: Use ordered-set aggregate where available; alternative: percentile_approx for large-scale.

---

### 29) Mode (most frequent value)

**Schema**: events(event_type STRING)

**Problem**: Return the most frequent event_type.

**Reference (SQL)**:
```sql

SELECT event_type
FROM (
  SELECT event_type, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS rn
  FROM events
  GROUP BY event_type
) t
WHERE rn = 1;
```
**Explanation**: Rank by count descending; break ties deterministically if required.

---

### 30) Stddev/variance

**Schema**: metrics(x DOUBLE)

**Problem**: Compute population stddev.

**Reference (SQL)**:
```sql

SELECT STDDEV_POP(x) AS stddev_pop
FROM metrics;
```
**Explanation**: Use POP vs SAMP variants appropriately for your statistical definition.

---

### 31) Row number per partition

**Schema**: employees(emp_id INT, dept STRING, salary INT)

**Problem**: Assign row numbers by dept ordered by salary desc.

**Reference (SQL)**:
```sql

SELECT emp_id, dept, salary,
       ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
FROM employees;
```
**Explanation**: ROW_NUMBER restarts for each partition.

---

### 32) Top 3 per group

**Schema**: employees(emp_id INT, dept STRING, salary INT)

**Problem**: Return top 3 highest paid employees in each dept.

**Reference (SQL)**:
```sql

SELECT * FROM (
  SELECT emp_id, dept, salary,
         ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
  FROM employees
) t
WHERE rn <= 3;
```
**Explanation**: Filter by row_number in an outer query for top-K per partition.

---

### 33) Dense rank with ties

**Schema**: scores(user_id INT, score INT)

**Problem**: Rank users by score allowing ties (dense).

**Reference (SQL)**:
```sql

SELECT user_id, score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS rnk
FROM scores;
```
**Explanation**: DENSE_RANK leaves no gaps in ranks for ties.

---

### 34) Nth value within group

**Schema**: orders(store STRING, amount INT)

**Problem**: Get 2nd highest amount per store.

**Reference (SQL)**:
```sql

SELECT store,
       NTH_VALUE(amount, 2) WITHIN GROUP (ORDER BY amount DESC) OVER (PARTITION BY store) AS second_highest
FROM orders;
```
**Explanation**: Ordered-set window function; alternative: row_number filter.

---

### 35) Running total by partition

**Schema**: daily_sales(store STRING, dt DATE, amount INT)

**Problem**: Cumulative sum per store by date.

**Reference (SQL)**:
```sql

SELECT store, dt, amount,
       SUM(amount) OVER (PARTITION BY store ORDER BY dt ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily_sales;
```
**Explanation**: Define frame for cumulative windows.

---

### 36) Moving average (7-day)

**Schema**: prices(ticker STRING, dt DATE, price DOUBLE)

**Problem**: 7-day moving average per ticker (row-based).

**Reference (SQL)**:
```sql

SELECT ticker, dt, price,
       AVG(price) OVER (PARTITION BY ticker ORDER BY dt ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma7
FROM prices;
```
**Explanation**: Row-based frame; for irregular dates, consider RANGE/intervals or calendar table join.

---

### 37) Lag/Lead differences

**Schema**: metrics(sensor STRING, ts TIMESTAMP, value DOUBLE)

**Problem**: Compute delta from previous reading per sensor.

**Reference (SQL)**:
```sql

SELECT sensor, ts, value,
       value - LAG(value) OVER (PARTITION BY sensor ORDER BY ts) AS delta
FROM metrics;
```
**Explanation**: LAG/LEAD compare a row to its neighbors.

---

### 38) First/Last value ignoring nulls

**Schema**: events(user_id INT, ts TIMESTAMP, attr STRING)

**Problem**: First non-null attr per user by time.

**Reference (SQL)**:
```sql

SELECT user_id,
       FIRST_VALUE(attr) IGNORE NULLS OVER (PARTITION BY user_id ORDER BY ts) AS first_attr
FROM events;
```
**Explanation**: Some dialects support IGNORE NULLS; otherwise use conditional aggregation.

---

### 39) Percentile within partition

**Schema**: latency(service STRING, ms DOUBLE)

**Problem**: P90 latency per service.

**Reference (SQL)**:
```sql

SELECT service,
       PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY ms) AS p90
FROM latency
GROUP BY service;
```
**Explanation**: Use ordered-set aggregate; in big data engines, consider approx_percentile.

---

### 40) Windowed distinct count (approx)

**Schema**: events(user_id INT, dt DATE)

**Problem**: Approx daily rolling 30-day DAU per product.

**Reference (SQL)**:
```sql

SELECT product, dt,
       APPROX_COUNT_DISTINCT(user_id) OVER (
         PARTITION BY product
         ORDER BY dt
         ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
       ) AS dau_30
FROM daily_events
-- assumes a pre-aggregated daily table daily_events(product, dt, user_id)
;
```
**Explanation**: Approx distinct per rolling window via windowed approx NDV (if supported).

---

### 41) Window frame vs default

**Schema**: payments(user_id INT, ts TIMESTAMP, amount INT)

**Problem**: Show difference between default RANGE and explicit ROWS frame (dialect-specific).

**Reference (SQL)**:
```sql

SELECT user_id, ts, amount,
       SUM(amount) OVER (PARTITION BY user_id ORDER BY ts)                AS default_frame,
       SUM(amount) OVER (PARTITION BY user_id ORDER BY ts ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_frame
FROM payments;
```
**Explanation**: Default frame semantics vary; explicit frames avoid surprises.

---

### 42) Sessionization via gaps

**Schema**: events(user_id INT, ts TIMESTAMP)

**Problem**: Start a new session if gap > 30 minutes, then number sessions per user.

**Reference (SQL)**:
```sql

WITH e AS (
  SELECT user_id, ts,
         CASE WHEN ts - LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) > INTERVAL '30' MINUTE
              OR LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) IS NULL
              THEN 1 ELSE 0 END AS new_sess
  FROM events
), s AS (
  SELECT user_id, ts,
         SUM(new_sess) OVER (PARTITION BY user_id ORDER BY ts ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_id
  FROM e
)
SELECT * FROM s;
```
**Explanation**: Mark new sessions on gap > threshold; cumulative sum creates session ids.

---

### 43) Nth row per group (no ordered-set function)

**Schema**: orders(store STRING, amount INT)

**Problem**: Return 2nd highest amount per store using ROW_NUMBER.

**Reference (SQL)**:
```sql

SELECT store, amount
FROM (
  SELECT store, amount,
         ROW_NUMBER() OVER (PARTITION BY store ORDER BY amount DESC) AS rn
  FROM orders
) x
WHERE rn = 2;
```
**Explanation**: Portable alternative when NTH_VALUE is unavailable.

---

### 44) Windowed ratio to total

**Schema**: sales(store STRING, amount INT)

**Problem**: Share of each sale relative to total per store.

**Reference (SQL)**:
```sql

SELECT store, amount,
       amount * 1.0 / SUM(amount) OVER (PARTITION BY store) AS share
FROM sales;
```
**Explanation**: Window sum used to compute per-row proportions.

---

### 45) De-dup keep latest by timestamp

**Schema**: accounts(id INT, email STRING, updated_at TIMESTAMP)

**Problem**: Pick latest record per id.

**Reference (SQL)**:
```sql

SELECT id, email, updated_at
FROM (
  SELECT a.*, ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC) AS rn
  FROM accounts a
) t
WHERE rn = 1;
```
**Explanation**: Deterministic de-duplication via row_number + filter.

---

### 46) Build a calendar table (recursive CTE)

**Problem**: Generate dates from 2026-01-01 to 2026-12-31.

**Reference (SQL)**:
```sql

WITH RECURSIVE cal(d) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT d + INTERVAL '1' DAY FROM cal WHERE d < DATE '2026-12-31'
)
SELECT * FROM cal;
```
**Explanation**: Recursive CTE to generate a date series; in warehouses use built-in sequence/generate_series where available.

---

### 47) Month start/end derivation

**Schema**: table: orders(order_date DATE)

**Problem**: Return month_start and month_end for each order_date.

**Reference (SQL)**:
```sql

SELECT order_date,
       DATE_TRUNC('month', order_date)            AS month_start,
       (DATE_TRUNC('month', order_date) + INTERVAL '1' MONTH - INTERVAL '1' DAY) AS month_end
FROM orders;
```
**Explanation**: DATE_TRUNC to floor; add one month then subtract a day for end.

---

### 48) Quarter extraction and labeling

**Schema**: events(dt DATE)

**Problem**: Return year-quarter label like 2026-Q1.

**Reference (SQL)**:
```sql

SELECT CONCAT(EXTRACT(YEAR FROM dt), '-Q', EXTRACT(QUARTER FROM dt)) AS yq
FROM events;
```
**Explanation**: Use EXTRACT to derive parts; CONCAT to label.

---

### 49) Last 30 days filter (inclusive)

**Schema**: log(ts TIMESTAMP)

**Problem**: Select rows with ts in the last 30 days from CURRENT_DATE.

**Reference (SQL)**:
```sql

SELECT *
FROM log
WHERE ts >= (CURRENT_DATE - INTERVAL '30' DAY)
  AND ts <  (CURRENT_DATE + INTERVAL '1' DAY);
```
**Explanation**: Use date boundary logic; adjust if ts is DATE vs TIMESTAMP.

---

### 50) Session length between first and last event

**Schema**: events(user_id INT, session_id INT, ts TIMESTAMP)

**Problem**: Compute session duration in minutes.

**Reference (SQL)**:
```sql

SELECT user_id, session_id,
       EXTRACT(EPOCH FROM (MAX(ts) - MIN(ts))) / 60.0 AS duration_min
FROM events
GROUP BY user_id, session_id;
```
**Explanation**: Subtract timestamps and convert to minutes (dialect-specific functions vary).

---

### 51) Week-of-year with ISO standard

**Schema**: events(dt DATE)

**Problem**: Return ISO week number.

**Reference (SQL)**:
```sql

SELECT EXTRACT(ISOWEEK FROM dt) AS iso_week
FROM events;
```
**Explanation**: Prefer ISO week to handle year boundaries consistently.

---

### 52) Monthly cohort assignment

**Schema**: users(user_id INT, signup_date DATE)

**Problem**: Assign cohort_month = first day of signup month.

**Reference (SQL)**:
```sql

SELECT user_id, DATE_TRUNC('month', signup_date) AS cohort_month
FROM users;
```
**Explanation**: DATE_TRUNC provides cohort buckets.

---

### 53) Active users per month (retention base)

**Schema**: events(user_id INT, dt DATE)

**Problem**: Distinct active users per month.

**Reference (SQL)**:
```sql

SELECT DATE_TRUNC('month', dt) AS month,
       COUNT(DISTINCT user_id) AS dau
FROM events
GROUP BY 1
ORDER BY 1;
```
**Explanation**: Aggregate on month buckets with DISTINCT users.

---

### 54) Sparsity fill with calendar join

**Schema**: daily_sales(dt DATE, amount INT)

**Problem**: Return all days and fill missing days with amount 0.

**Reference (SQL)**:
```sql

WITH RECURSIVE cal(d) AS (
  SELECT MIN(dt) FROM daily_sales
  UNION ALL
  SELECT d + INTERVAL '1' DAY FROM cal WHERE d < (SELECT MAX(dt) FROM daily_sales)
)
SELECT c.d AS dt, COALESCE(s.amount, 0) AS amount
FROM cal c
LEFT JOIN daily_sales s ON s.dt = c.d
ORDER BY dt;
```
**Explanation**: Calendar scaffold + left join to fill missing dates.

---

### 55) End-of-month snapshot

**Schema**: balances(account_id INT, dt DATE, balance NUMERIC)

**Problem**: Pick the last known balance per account per month.

**Reference (SQL)**:
```sql

SELECT account_id, month, balance
FROM (
  SELECT account_id,
         DATE_TRUNC('month', dt) AS month,
         balance,
         ROW_NUMBER() OVER (PARTITION BY account_id, DATE_TRUNC('month', dt) ORDER BY dt DESC) AS rn
  FROM balances
) t
WHERE rn = 1;
```
**Explanation**: Row-number choose latest row within the month partition.

---

### 56) Split and normalize emails

**Schema**: emails(raw STRING)

**Problem**: Lowercase, trim, and extract domain part.

**Reference (SQL)**:
```sql

SELECT LOWER(TRIM(raw))                      AS email_norm,
       SPLIT_PART(LOWER(TRIM(raw)), '@', 2)  AS domain
FROM emails;
```
**Explanation**: SPLIT_PART or SUBSTRING/REGEXP to parse domains.

---

### 57) Extract JSON field

**Schema**: events(payload JSON)

**Problem**: Get user.id from JSON payload.

**Reference (SQL)**:
```sql

SELECT JSON_VALUE(payload, '$.user.id') AS user_id
FROM events;
```
**Explanation**: Use JSON_VALUE/EXTRACT/GET depending on dialect.

---

### 58) Flatten JSON array

**Schema**: events(payload JSON)

**Problem**: Explode items[] from payload per row.

**Reference (SQL)**:
```sql

SELECT e.*, j.value AS item
FROM events e,
LATERAL JSON_TABLE(e.payload, '$.items[*]' COLUMNS(value VARCHAR(200) PATH '$')) j;
```
**Explanation**: JSON_TABLE or UNNEST/EXPLODE based on engine.

---

### 59) Regex extract

**Schema**: logs(line STRING)

**Problem**: Extract first IPv4 address.

**Reference (SQL)**:
```sql

SELECT REGEXP_SUBSTR(line, '(\d{1,3}\.){3}\d{1,3}') AS ip
FROM logs;
```
**Explanation**: REGEXP_SUBSTR captures the first match; escape backslashes in patterns.

---

### 60) Concatenate with delimiter

**Schema**: names(first STRING, last STRING)

**Problem**: Return full_name = last, first.

**Reference (SQL)**:
```sql

SELECT CONCAT(last, ', ', first) AS full_name
FROM names;
```
**Explanation**: Use CONCAT/|| depending on dialect.

---

### 61) Array length and element access

**Schema**: arr_tbl(id INT, tags ARRAY<STRING>)

**Problem**: Return id, array_size, first_tag.

**Reference (SQL)**:
```sql

SELECT id, CARDINALITY(tags) AS array_size, tags[1] AS first_tag
FROM arr_tbl;
```
**Explanation**: Array functions vary; many engines are 1-based for subscripts.

---

### 62) Aggregate to JSON array

**Schema**: orders(order_id INT, item STRING)

**Problem**: Group items per order as JSON array.

**Reference (SQL)**:
```sql

SELECT order_id, JSON_ARRAYAGG(item) AS items
FROM orders
GROUP BY order_id;
```
**Explanation**: Use JSON array aggregation to serialize rows.

---

### 63) String normalization for join keys

**Schema**: src(name STRING), dim(name STRING)

**Problem**: Join using cleaned names (lowercase, spaces to underscore).

**Reference (SQL)**:
```sql

SELECT *
FROM (
  SELECT LOWER(REPLACE(TRIM(name), ' ', '_')) AS k FROM src
) s
JOIN (
  SELECT LOWER(REPLACE(TRIM(name), ' ', '_')) AS k FROM dim
) d ON d.k = s.k;
```
**Explanation**: Normalize both sides identically to improve match rate.

---

### 64) Find duplicates by natural key

**Schema**: users(email STRING)

**Problem**: Return emails that appear more than once.

**Reference (SQL)**:
```sql

SELECT email, COUNT(*) AS cnt
FROM users
GROUP BY email
HAVING COUNT(*) > 1;
```
**Explanation**: Detect dupes for remediation or survivorship rules.

---

### 65) Keep latest record per natural key

**Schema**: users(email STRING, updated_at TIMESTAMP, payload JSON)

**Problem**: Survive duplicates keeping the latest row.

**Reference (SQL)**:
```sql

SELECT email, updated_at, payload
FROM (
  SELECT u.*, ROW_NUMBER() OVER (PARTITION BY email ORDER BY updated_at DESC) AS rn
  FROM users u
) t
WHERE rn = 1;
```
**Explanation**: Row-number + filter is deterministic and scalable.

---

### 66) Generate surrogate key (hash)

**Schema**: dim_customer(natural_key STRING, attr1 STRING, attr2 STRING)

**Problem**: Create deterministic surrogate key.

**Reference (SQL)**:
```sql

SELECT MD5(CONCAT(natural_key, '||', attr1, '||', attr2)) AS sk,
       *
FROM dim_customer;
```
**Explanation**: Hash of natural attributes yields stable surrogate; mind collision risk.

---

### 67) SCD Type 1 (overwrite)

**Schema**: staging(k STRING, v STRING); dim(k STRING, v STRING)

**Problem**: Bring latest values from staging into dim (overwrite).

**Reference (SQL)**:
```sql

-- Pseudo-SQL: in warehouses use MERGE
MERGE INTO dim d
USING staging s
ON d.k = s.k
WHEN MATCHED THEN UPDATE SET d.v = s.v
WHEN NOT MATCHED THEN INSERT (k, v) VALUES (s.k, s.v);
```
**Explanation**: Type 1 overwrites existing attribute values.

---

### 68) SCD Type 2 (history tracking)

**Schema**: staging(k STRING, v STRING); dim(k STRING, v STRING, eff_start DATE, eff_end DATE, is_current BOOLEAN)

**Problem**: Close current row and insert a new current row when value changes.

**Reference (SQL)**:
```sql

-- Typical two-step approach; warehouses support MERGE variants
-- 1) Close out existing current rows where value changes
UPDATE dim d
SET eff_end = CURRENT_DATE, is_current = FALSE
WHERE is_current = TRUE
  AND EXISTS (
    SELECT 1 FROM staging s WHERE s.k = d.k AND s.v <> d.v
  );

-- 2) Insert new current rows
INSERT INTO dim(k, v, eff_start, eff_end, is_current)
SELECT s.k, s.v, CURRENT_DATE, DATE '9999-12-31', TRUE
FROM staging s
LEFT JOIN dim d ON d.k = s.k AND d.is_current = TRUE
WHERE d.k IS NULL OR d.v <> s.v;
```
**Explanation**: Close the current record (set eff_end) then insert a new row. In Delta/BigQuery/Redshift use MERGE to combine steps.

---

### 69) Data conformance checks

**Schema**: orders(order_id INT, amount NUMERIC, currency STRING)

**Problem**: Flag rows failing sanity checks (amount<=0 or currency not in list).

**Reference (SQL)**:
```sql

SELECT order_id,
       CASE WHEN amount <= 0 THEN 'NEG_OR_ZERO' END AS amount_issue,
       CASE WHEN currency NOT IN ('USD','EUR','INR') THEN 'BAD_CCY' END AS currency_issue
FROM orders
WHERE amount <= 0 OR currency NOT IN ('USD','EUR','INR');
```
**Explanation**: Emit rule IDs to drive DQ dashboards.

---

### 70) Survivorship rule with COALESCE and MAX_BY

**Schema**: profiles(user_id INT, email STRING, updated_at TIMESTAMP)

**Problem**: Pick the email from the most recent record per user.

**Reference (SQL)**:
```sql

SELECT user_id,
       MAX_BY(email, updated_at) AS chosen_email
FROM profiles
GROUP BY user_id;
```
**Explanation**: MAX_BY (Databricks/Presto) picks value with max ordering key; else use row_number.

---

### 71) Find orphan facts

**Schema**: fact_sales(order_id INT, product_id INT), dim_product(product_id INT)

**Problem**: Fact rows referencing missing product dimension.

**Reference (SQL)**:
```sql

SELECT f.*
FROM fact_sales f
LEFT JOIN dim_product d ON d.product_id = f.product_id
WHERE d.product_id IS NULL;
```
**Explanation**: Identifies referential integrity violations.

---

### 72) Checksum for change detection

**Schema**: staging(k STRING, v1 STRING, v2 STRING)

**Problem**: Compute row checksum to detect changes.

**Reference (SQL)**:
```sql

SELECT k,
       MD5(CONCAT_WS('||', COALESCE(v1,''), COALESCE(v2,''))) AS row_hash
FROM staging;
```
**Explanation**: Row hash simplifies CDC comparisons and idempotent loads.

---

### 73) Null-safe equality

**Schema**: a(id INT, v STRING), b(id INT, v STRING)

**Problem**: Match rows where v is equal treating NULL=NULL as true.

**Reference (SQL)**:
```sql

SELECT *
FROM a
JOIN b ON a.id = b.id AND (a.v = b.v OR (a.v IS NULL AND b.v IS NULL));
```
**Explanation**: ANSI SQL lacks <=>; emulate with explicit NULL checks.

---

### 74) Pivot rows to columns

**Schema**: sales(store STRING, category STRING, amount INT)

**Problem**: Revenue by category as columns per store.

**Reference (SQL)**:
```sql

SELECT *
FROM (
  SELECT store, category, amount FROM sales
) s
PIVOT (
  SUM(amount) FOR category IN ('A','B','C')
) p;
```
**Explanation**: Pivot syntax varies widely; alternative is conditional aggregation.

---

### 75) Unpivot columns to rows

**Schema**: metrics(dt DATE, revenue INT, cost INT, profit INT)

**Problem**: Turn wide metrics into (metric, value) rows.

**Reference (SQL)**:
```sql

SELECT dt, metric, value
FROM metrics
UNPIVOT (value FOR metric IN (revenue, cost, profit)) AS u;
```
**Explanation**: UNPIVOT or use UNION ALL/STACK constructs.

---

### 76) Recursive hierarchy (org chart)

**Schema**: employees(emp_id INT, manager_id INT, name STRING)

**Problem**: Return chain of command for each employee.

**Reference (SQL)**:
```sql

WITH RECURSIVE org(emp_id, manager_id, name, depth, path) AS (
  SELECT emp_id, manager_id, name, 0 AS depth, CAST(emp_id AS VARCHAR) AS path
  FROM employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.emp_id, e.manager_id, e.name, o.depth+1,
         CONCAT(o.path, '>', e.emp_id)
  FROM employees e
  JOIN org o ON o.emp_id = e.manager_id
)
SELECT * FROM org;
```
**Explanation**: Recursive CTE walks parent→child to build hierarchy and depth.

---

### 77) Gaps and islands (consecutive days)

**Schema**: attendance(emp_id INT, day DATE)

**Problem**: Identify contiguous attendance streaks.

**Reference (SQL)**:
```sql

SELECT emp_id, MIN(day) AS start_day, MAX(day) AS end_day
FROM (
  SELECT emp_id, day,
         (day - INTERVAL '1' DAY) - LAG(day) OVER (PARTITION BY emp_id ORDER BY day) AS gap,
         SUM(CASE WHEN LAG(day) OVER (PARTITION BY emp_id ORDER BY day) = day - INTERVAL '1' DAY THEN 0 ELSE 1 END)
           OVER (PARTITION BY emp_id ORDER BY day) AS island_id
  FROM attendance
) t
GROUP BY emp_id, island_id
ORDER BY emp_id, start_day;
```
**Explanation**: Detect breakpoints then group by a running island id.

---

### 78) Top-N per group with ties (dense_rank)

**Schema**: movies(genre STRING, title STRING, rating DOUBLE)

**Problem**: Return all movies in each genre that tie within top 3 ratings.

**Reference (SQL)**:
```sql

SELECT * FROM (
  SELECT genre, title, rating,
         DENSE_RANK() OVER (PARTITION BY genre ORDER BY rating DESC) AS r
  FROM movies
) x
WHERE r <= 3;
```
**Explanation**: DENSE_RANK preserves ties.

---

### 79) Interval overlap detection

**Schema**: meetings(room STRING, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Find meetings that overlap in the same room.

**Reference (SQL)**:
```sql

SELECT m1.*
FROM meetings m1
JOIN meetings m2
  ON m1.room = m2.room
 AND m1.start_ts < m2.end_ts
 AND m1.end_ts   > m2.start_ts
WHERE m1.start_ts < m2.start_ts;
```
**Explanation**: Proper overlap condition: start < other_end AND end > other_start.

---

### 80) Find median per group (approx)

**Schema**: orders(store STRING, amount DOUBLE)

**Problem**: Approximate median per store.

**Reference (SQL)**:
```sql

SELECT store, APPROX_PERCENTILE(amount, 0.5) AS median
FROM orders
GROUP BY store;
```
**Explanation**: Approx percentile functions scale better for big data.

---

### 81) Cumulative distinct count (monthly)

**Schema**: events(user_id INT, dt DATE)

**Problem**: Distinct users seen up to each month.

**Reference (SQL)**:
```sql

WITH m AS (
  SELECT DATE_TRUNC('month', dt) AS month, user_id
  FROM events
  GROUP BY 1, user_id
)
SELECT month,
       COUNT(DISTINCT user_id) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_users
FROM (
  SELECT DISTINCT month FROM m
) months
LEFT JOIN m USING (month)
ORDER BY month;
```
**Explanation**: De-duplicate within month then window distinct (engine-dependent; some require workarounds).

---

### 82) Percent-of-total across entire table

**Schema**: transactions(amount NUMERIC)

**Problem**: Return each transaction and its percent contribution to total.

**Reference (SQL)**:
```sql

WITH t AS (
  SELECT amount FROM transactions
)
SELECT amount,
       amount * 100.0 / SUM(amount) OVER () AS pct_of_total
FROM t;
```
**Explanation**: Window with empty OVER() frame computes table-level totals for each row.

---

### 83) Best practice note #83

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 84) Best practice note #84

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 85) Best practice note #85

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 86) Best practice note #86

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 87) Best practice note #87

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 88) Best practice note #88

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 89) Best practice note #89

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 90) Best practice note #90

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 91) Best practice note #91

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 92) Best practice note #92

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 93) Best practice note #93

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 94) Best practice note #94

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 95) Best practice note #95

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 96) Best practice note #96

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 97) Best practice note #97

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 98) Best practice note #98

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 99) Best practice note #99

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---

### 100) Best practice note #100

**Problem**: Explain why half-open intervals [start, end) are preferred for date ranges.

**Reference (SQL)**:
```sql

-- Use start inclusive and end exclusive to avoid overlap and simplify chaining
-- WHERE dt >= :start AND dt < :end
```
**Explanation**: Prevents double-counting boundaries and eases partition elimination.

---



# Classic SQL Interview Problems: 100 Logic & Scenario Q&A

> Each problem is language-agnostic in spirit, with an **ANSI SQL** reference solution and a concise **explanation**. Adjust minor syntax for your SQL engine (e.g., BigQuery, Postgres, Redshift, Snowflake, Databricks SQL).

---

### 1) Top-3 Salaries per Department (include ties)

**Schema**: employees(emp_id INT, dept STRING, salary INT)

**Problem**: Return all employees whose salary ranks within the top 3 in their department (ties included).

**Reference (SQL)**:
```sql
SELECT *
FROM (
  SELECT emp_id, dept, salary,
         DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS r
  FROM employees
) t
WHERE r <= 3;
```
**Explanation**: Use DENSE_RANK so equal salaries get the same rank; filter ranks ≤ 3 to include ties.

---

### 2) Top-3 Salaries per Department (break ties)

**Schema**: employees(emp_id INT, dept STRING, salary INT)

**Problem**: Return exactly three employees per department by salary (break ties arbitrarily but deterministically).

**Reference (SQL)**:
```sql
SELECT *
FROM (
  SELECT emp_id, dept, salary,
         ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC, emp_id) AS rn
  FROM employees
) t
WHERE rn <= 3;
```
**Explanation**: ROW_NUMBER breaks ties using a secondary key; ensures exactly 3 rows per department when available.

---

### 3) Nth Highest Salary (distinct values)

**Schema**: employees(emp_id INT, salary INT)

**Problem**: Return the 5th highest distinct salary.

**Reference (SQL)**:
```sql
SELECT salary
FROM (
  SELECT DISTINCT salary,
         DENSE_RANK() OVER (ORDER BY salary DESC) AS r
  FROM employees
) t
WHERE r = 5;
```
**Explanation**: Rank distinct salaries with DENSE_RANK and pick r = N.

---

### 4) Employees Earning Above Department Average

**Schema**: employees(emp_id INT, dept STRING, salary INT)

**Problem**: List employees whose salary is strictly above their department average.

**Reference (SQL)**:
```sql
SELECT emp_id, dept, salary
FROM (
  SELECT emp_id, dept, salary,
         AVG(salary) OVER (PARTITION BY dept) AS dept_avg
  FROM employees
) t
WHERE salary > dept_avg;
```
**Explanation**: Compute departmental average by window, filter rows above it.

---

### 5) Percent Rank of Salary Within Department

**Schema**: employees(emp_id INT, dept STRING, salary INT)

**Problem**: Compute PERCENT_RANK of each employee salary within department.

**Reference (SQL)**:
```sql
SELECT emp_id, dept, salary,
       PERCENT_RANK() OVER (PARTITION BY dept ORDER BY salary) AS pct_rank
FROM employees;
```
**Explanation**: PERCENT_RANK normalizes rank within [0,1]; equal values share ties.

---

### 6) Cumulative Sales by Store Date

**Schema**: daily_sales(store STRING, dt DATE, amount INT)

**Problem**: Return cumulative sum of sales per store ordered by date.

**Reference (SQL)**:
```sql
SELECT store, dt, amount,
       SUM(amount) OVER (
           PARTITION BY store
           ORDER BY dt
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM daily_sales;
```
**Explanation**: Explicit ROWS frame defines a standard cumulative sum per store.

---

### 7) 7-Day Moving Average Price per Ticker

**Schema**: prices(ticker STRING, dt DATE, price DOUBLE)

**Problem**: Compute 7-row moving average of price for each ticker ordered by date.

**Reference (SQL)**:
```sql
SELECT ticker, dt, price,
       AVG(price) OVER (
           PARTITION BY ticker
           ORDER BY dt
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ) AS ma7
FROM prices;
```
**Explanation**: Row-based window of 7 rows (current + previous 6); for gaps in calendar consider range/calendar join.

---

### 8) Daily Delta from Previous Reading

**Schema**: metrics(sensor STRING, ts TIMESTAMP, value DOUBLE)

**Problem**: Compute difference between current and previous reading per sensor.

**Reference (SQL)**:
```sql
SELECT sensor, ts, value,
       value - LAG(value) OVER (PARTITION BY sensor ORDER BY ts) AS delta
FROM metrics;
```
**Explanation**: LAG reads prior value in the same partition ordered by timestamp.

---

### 9) First Non-Null Attribute by Time (per user)

**Schema**: events(user_id INT, ts TIMESTAMP, attr STRING)

**Problem**: Return the first non-null attr seen per user as of each row.

**Reference (SQL)**:
```sql
SELECT user_id, ts, attr,
       FIRST_VALUE(attr) IGNORE NULLS OVER (PARTITION BY user_id ORDER BY ts) AS first_attr
FROM events;
```
**Explanation**: IGNORE NULLS is supported in several engines; otherwise emulate with conditional aggregation.

---

### 10) Share of Sale vs. Total in Store

**Schema**: sales(store STRING, amount INT)

**Problem**: Compute each sale’s fraction of the total amount within its store.

**Reference (SQL)**:
```sql
SELECT store, amount,
       amount * 1.0 / SUM(amount) OVER (PARTITION BY store) AS share
FROM sales;
```
**Explanation**: Window sum over store used as denominator for each row’s share.

---

### 11) Latest Row per Account (De-dup)

**Schema**: accounts(id INT, email STRING, updated_at TIMESTAMP)

**Problem**: For each account id, keep only the most recent record.

**Reference (SQL)**:
```sql
SELECT id, email, updated_at
FROM (
  SELECT a.*, ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC) AS rn
  FROM accounts a
) t
WHERE rn = 1;
```
**Explanation**: ROW_NUMBER per key ordered by updated_at descending selects the latest row.

---

### 12) First Order Date per Customer

**Schema**: orders(order_id INT, customer_id INT, order_date DATE)

**Problem**: Return the first order date for each customer.

**Reference (SQL)**:
```sql
SELECT customer_id, MIN(order_date) AS first_order
FROM orders
GROUP BY customer_id;
```
**Explanation**: Simple aggregate MIN per customer finds first order date.

---

### 13) Consecutive ID Ranges

**Schema**: ids(id INT)

**Problem**: Group consecutive integer ids into [start_id, end_id] ranges.

**Reference (SQL)**:
```sql
WITH x AS (
  SELECT id, id - ROW_NUMBER() OVER (ORDER BY id) AS grp
  FROM ids
)
SELECT MIN(id) AS start_id, MAX(id) AS end_id
FROM x
GROUP BY grp
ORDER BY start_id;
```
**Explanation**: Subtracting row_number creates identical grp for consecutive runs (islands).

---

### 14) Attendance Streaks ≥ 3 Days

**Schema**: attendance(emp_id INT, day DATE)

**Problem**: Find employees who have at least 3 consecutive attendance days.

**Reference (SQL)**:
```sql
WITH x AS (
  SELECT emp_id, day,
         day - ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY day) * INTERVAL '1' DAY AS grp
  FROM attendance
)
SELECT emp_id
FROM x
GROUP BY emp_id, grp
HAVING COUNT(*) >= 3;
```
**Explanation**: Use day minus row_number * 1 day to align consecutive days to a constant key.

---

### 15) Find Missing Dates

**Schema**: daily(dt DATE)

**Problem**: List dates missing between MIN(dt) and MAX(dt).

**Reference (SQL)**:
```sql
WITH RECURSIVE cal(d) AS (
  SELECT (SELECT MIN(dt) FROM daily)
  UNION ALL
  SELECT d + INTERVAL '1' DAY FROM cal WHERE d < (SELECT MAX(dt) FROM daily)
)
SELECT c.d AS missing_date
FROM cal c
LEFT JOIN daily d ON d.dt = c.d
WHERE d.dt IS NULL;
```
**Explanation**: Calendar scaffold vs. actual table reveals gaps.

---

### 16) Sessions by 30-Minute Gap

**Schema**: events(user_id INT, ts TIMESTAMP)

**Problem**: Start a new session when the gap between events exceeds 30 minutes.

**Reference (SQL)**:
```sql
WITH e AS (
  SELECT user_id, ts,
         CASE WHEN LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) IS NULL
                   OR ts - LAG(ts) OVER (PARTITION BY user_id ORDER BY ts) > INTERVAL '30' MINUTE
              THEN 1 ELSE 0 END AS new_flag
  FROM events
), s AS (
  SELECT user_id, ts,
         SUM(new_flag) OVER (PARTITION BY user_id ORDER BY ts) AS session_id
  FROM e
)
SELECT * FROM s;
```
**Explanation**: A gap greater than threshold marks a new session; cumulative sum numbers sessions.

---

### 17) Longest Consecutive Workdays per Employee

**Schema**: work(emp_id INT, day DATE)

**Problem**: Return the maximum streak length of consecutive working days per employee.

**Reference (SQL)**:
```sql
WITH x AS (
  SELECT emp_id, day,
         day - ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY day) * INTERVAL '1' DAY AS grp
  FROM work
)
SELECT emp_id, MAX(COUNT(*)) AS max_streak
FROM x
GROUP BY emp_id, grp;
```
**Explanation**: Group consecutive days into islands and count; take max per employee.

---

### 18) Find Duplicated Order Numbers

**Schema**: orders(order_no STRING)

**Problem**: Return order_no values that appear more than once.

**Reference (SQL)**:
```sql
SELECT order_no, COUNT(*) AS cnt
FROM orders
GROUP BY order_no
HAVING COUNT(*) > 1;
```
**Explanation**: Basic duplicate detection using COUNT and HAVING.

---

### 19) Consecutive Value Increase (3 months)

**Schema**: m_sales(customer_id INT, month DATE, total NUMERIC)

**Problem**: Find customers with strictly increasing totals for 3 consecutive months.

**Reference (SQL)**:
```sql
WITH r AS (
  SELECT customer_id, month, total,
         total > LAG(total) OVER (PARTITION BY customer_id ORDER BY month)   AS up1,
         LAG(total) OVER (PARTITION BY customer_id ORDER BY month) >
         LAG(total,2) OVER (PARTITION BY customer_id ORDER BY month)        AS up2
  FROM m_sales
)
SELECT DISTINCT customer_id
FROM r
WHERE up1 AND up2;
```
**Explanation**: Compare successive months with LAG to detect two successive increases (3-month rising window).

---

### 20) Detect Price Changes Day-over-Day

**Schema**: prices(sku STRING, dt DATE, price NUMERIC)

**Problem**: Return rows where today’s price is different from previous day for the same SKU.

**Reference (SQL)**:
```sql
SELECT p.*
FROM prices p
LEFT JOIN prices prev
  ON prev.sku = p.sku AND prev.dt = p.dt - INTERVAL '1' DAY
WHERE prev.price IS NULL OR p.price <> prev.price;
```
**Explanation**: Self-join on date offset; include first day as change vs. NULL.

---

### 21) Rank Dense vs. Row_Number Example

**Schema**: scores(user STRING, score INT)

**Problem**: Show the difference between DENSE_RANK and ROW_NUMBER over the same ordering.

**Reference (SQL)**:
```sql
SELECT user, score,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dr,
       ROW_NUMBER() OVER (ORDER BY score DESC, user) AS rn
FROM scores;
```
**Explanation**: Dense rank gives same rank on ties; row-number always increments and can break ties with a secondary key.

---

### 22) Monthly Active Users (MAU)

**Schema**: events(user_id INT, dt DATE)

**Problem**: Return distinct active users per month.

**Reference (SQL)**:
```sql
SELECT DATE_TRUNC('month', dt) AS month,
       COUNT(DISTINCT user_id) AS mau
FROM events
GROUP BY 1
ORDER BY 1;
```
**Explanation**: Aggregate on month buckets with DISTINCT users.

---

### 23) Channel Split with CASE

**Schema**: sales(channel STRING, amount NUMERIC)

**Problem**: Compute total per channel and overall in one query.

**Reference (SQL)**:
```sql
SELECT SUM(amount) AS total,
       SUM(CASE WHEN channel = 'online'  THEN amount ELSE 0 END) AS online,
       SUM(CASE WHEN channel = 'store'   THEN amount ELSE 0 END) AS store,
       SUM(CASE WHEN channel = 'partner' THEN amount ELSE 0 END) AS partner
FROM sales;
```
**Explanation**: Conditional aggregation with CASE lets you pivot in-place.

---

### 24) Customers with High Lifetime Value

**Schema**: orders(customer_id INT, amount NUMERIC)

**Problem**: List customers with lifetime spend > 5000.

**Reference (SQL)**:
```sql
SELECT customer_id, SUM(amount) AS lifetime_spend
FROM orders
GROUP BY customer_id
HAVING SUM(amount) > 5000;
```
**Explanation**: HAVING filters post-aggregation; WHERE would filter pre-aggregation.

---

### 25) Mode (Most Frequent Event)

**Schema**: events(event_type STRING)

**Problem**: Return the most common event_type.

**Reference (SQL)**:
```sql
SELECT event_type
FROM (
  SELECT event_type,
         ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, event_type) AS rn
  FROM events
  GROUP BY event_type
) t
WHERE rn = 1;
```
**Explanation**: Rank groups by count; break ties deterministically.

---

### 26) Median Payment (Exact Ordered-Set)

**Schema**: payments(amount NUMERIC)

**Problem**: Compute exact median using ordered-set aggregate if supported.

**Reference (SQL)**:
```sql
SELECT PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY amount) AS median
FROM payments;
```
**Explanation**: PERCENTILE_DISC returns an actual value from the set; alternative is approximate percentile for big data.

---

### 27) ROLLUP Store→Category→Total

**Schema**: sales(store STRING, category STRING, amount NUMERIC)

**Problem**: Compute totals by (store,category), by store, and grand total.

**Reference (SQL)**:
```sql
SELECT store, category, SUM(amount) AS total
FROM sales
GROUP BY ROLLUP (store, category);
```
**Explanation**: ROLLUP yields hierarchical subtotals; identify totals via GROUPING() if needed.

---

### 28) CUBE Region×Channel

**Schema**: sales(region STRING, channel STRING, amount NUMERIC)

**Problem**: Compute totals for all combinations of region and channel.

**Reference (SQL)**:
```sql
SELECT region, channel, SUM(amount) AS total
FROM sales
GROUP BY CUBE (region, channel);
```
**Explanation**: CUBE produces all cross-subtotals; may be expensive—use sparingly.

---

### 29) GROUPING SETS: Region-only and Product-only

**Schema**: sales(region STRING, product STRING, amount NUMERIC)

**Problem**: Produce totals by region and totals by product without cross totals.

**Reference (SQL)**:
```sql
SELECT region, product, SUM(amount) AS total
FROM sales
GROUP BY GROUPING SETS ((region), (product));
```
**Explanation**: GROUPING SETS allows arbitrary subtotal combinations.

---

### 30) Population Stddev

**Schema**: metrics(x DOUBLE)

**Problem**: Compute population standard deviation.

**Reference (SQL)**:
```sql
SELECT STDDEV_POP(x) AS stddev_pop
FROM metrics;
```
**Explanation**: Use POP vs SAMP depending on your statistical need.

---

### 31) Customers with ≥ 10 Orders

**Schema**: orders(customer_id INT)

**Problem**: Return customer_ids with at least 10 orders.

**Reference (SQL)**:
```sql
SELECT customer_id
FROM orders
GROUP BY customer_id
HAVING COUNT(*) >= 10;
```
**Explanation**: Threshold-based filtering after aggregation.

---

### 32) VIP Orders via EXISTS (Semi-Join)

**Schema**: orders(order_id INT, customer_id INT), vip(customer_id INT)

**Problem**: Return orders placed by VIP customers.

**Reference (SQL)**:
```sql
SELECT o.*
FROM orders o
WHERE EXISTS (
  SELECT 1 FROM vip v WHERE v.customer_id = o.customer_id
);
```
**Explanation**: EXISTS often optimizes better than IN for correlated existence checks.

---

### 33) Left Anti-Join (Customers without Orders)

**Schema**: customers(cust_id INT), orders(order_id INT, cust_id INT)

**Problem**: List customers who have never placed an order.

**Reference (SQL)**:
```sql
SELECT c.*
FROM customers c
LEFT JOIN orders o ON o.cust_id = c.cust_id
WHERE o.cust_id IS NULL;
```
**Explanation**: Left join + NULL on right yields anti-join set.

---

### 34) Full Outer Join for CDC Diff

**Schema**: src(key INT, val STRING), tgt(key INT, val STRING)

**Problem**: Identify inserts, updates, and deletes between src and tgt.

**Reference (SQL)**:
```sql
SELECT COALESCE(s.key, t.key) AS key,
       s.val AS src_val,
       t.val AS tgt_val,
       CASE
         WHEN s.key IS NOT NULL AND t.key IS NULL THEN 'INSERT'
         WHEN s.key IS NULL AND t.key IS NOT NULL THEN 'DELETE'
         WHEN s.val <> t.val THEN 'UPDATE'
         ELSE 'UNCHANGED'
       END AS change_type
FROM src s
FULL OUTER JOIN tgt t ON t.key = s.key;
```
**Explanation**: FULL OUTER surfaces all rows to classify changes.

---

### 35) Composite Key Join

**Schema**: a(k1 INT, k2 INT, v INT), b(k1 INT, k2 INT, x INT)

**Problem**: Join tables on a composite key (k1,k2).

**Reference (SQL)**:
```sql
SELECT *
FROM a
JOIN b ON a.k1 = b.k1 AND a.k2 = b.k2;
```
**Explanation**: All key columns must match.

---

### 36) Set Difference via EXCEPT/MINUS

**Schema**: events_a(e STRING), events_b(e STRING)

**Problem**: Return events present in A but not in B.

**Reference (SQL)**:
```sql
SELECT e FROM events_a
EXCEPT
SELECT e FROM events_b;
```
**Explanation**: Use EXCEPT (or MINUS) for set difference on distinct rows.

---

### 37) Join with Date Range (SCD2 Lookup)

**Schema**: fact(order_id INT, order_date DATE, region_id INT), dim_region(region_id INT, region STRING, eff_start DATE, eff_end DATE)

**Problem**: Attach the region name valid at order_date.

**Reference (SQL)**:
```sql
SELECT f.order_id, d.region
FROM fact f
JOIN dim_region d
  ON d.region_id = f.region_id
 AND f.order_date >= d.eff_start
 AND f.order_date <  d.eff_end;
```
**Explanation**: Half-open intervals [start,end) avoid overlaps.

---

### 38) Self-Join to Compare Adjacent Days

**Schema**: prices(sku STRING, dt DATE, price NUMERIC)

**Problem**: Find dates when a SKU price increased vs previous day.

**Reference (SQL)**:
```sql
SELECT p1.sku, p1.dt
FROM prices p1
JOIN prices p0
  ON p0.sku = p1.sku AND p0.dt = p1.dt - INTERVAL '1' DAY
WHERE p1.price > p0.price;
```
**Explanation**: Offset join on prior day detects increases.

---

### 39) UNION ALL vs UNION

**Schema**: x(a INT), y(a INT)

**Problem**: Stack two tables and keep duplicates.

**Reference (SQL)**:
```sql
SELECT a FROM x
UNION ALL
SELECT a FROM y;
```
**Explanation**: UNION ALL avoids the DISTINCT step of UNION; faster when duplicates allowed.

---

### 40) Normalize Undirected Pairs

**Schema**: friendship(a INT, b INT)

**Problem**: Return unique undirected edges (a,b) with a<b.

**Reference (SQL)**:
```sql
SELECT LEAST(a,b) AS u, GREATEST(a,b) AS v
FROM friendship
GROUP BY LEAST(a,b), GREATEST(a,b);
```
**Explanation**: Use LEAST/GREATEST to canonicalize endpoints.

---

### 41) Orders without Matching Products

**Schema**: fact_sales(order_id INT, product_id INT), dim_product(product_id INT)

**Problem**: Find fact rows referencing missing dimension keys.

**Reference (SQL)**:
```sql
SELECT f.*
FROM fact_sales f
LEFT JOIN dim_product d ON d.product_id = f.product_id
WHERE d.product_id IS NULL;
```
**Explanation**: Referential integrity check via left anti-join.

---

### 42) Overlapping Meetings in Same Room

**Schema**: meetings(room STRING, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Find meetings that overlap within the same room.

**Reference (SQL)**:
```sql
SELECT m1.*
FROM meetings m1
JOIN meetings m2
  ON m1.room = m2.room
 AND m1.start_ts < m2.end_ts
 AND m1.end_ts   > m2.start_ts
WHERE m1.start_ts < m2.start_ts;
```
**Explanation**: Proper overlap condition: start < other_end AND end > other_start.

---

### 43) Minimum Rooms Needed (concept via sweep)

**Schema**: meetings(start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Compute minimal number of rooms required for all meetings (illustrative SQL).

**Reference (SQL)**:
```sql
WITH evt AS (
  SELECT start_ts AS t, 1 AS d FROM meetings
  UNION ALL
  SELECT end_ts   AS t, -1 AS d FROM meetings
), run AS (
  SELECT t,
         SUM(d) OVER (ORDER BY t ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS in_use
  FROM evt
)
SELECT MAX(in_use) AS min_rooms
FROM run;
```
**Explanation**: Line sweep counts concurrent meetings; maximum concurrent equals room count.

---

### 44) Find Idle Gaps Between Tasks

**Schema**: tasks(machine STRING, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Return idle intervals per machine between adjacent tasks.

**Reference (SQL)**:
```sql
SELECT machine,
       prev_end AS idle_start,
       start_ts AS idle_end
FROM (
  SELECT machine, start_ts, end_ts,
         LAG(end_ts) OVER (PARTITION BY machine ORDER BY start_ts) AS prev_end
  FROM tasks
) t
WHERE prev_end IS NOT NULL AND start_ts > prev_end;
```
**Explanation**: Use LAG to access previous end and compute gap.

---

### 45) Find Overbooked Employees

**Schema**: shift(emp_id INT, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Employees with overlapping shifts.

**Reference (SQL)**:
```sql
SELECT DISTINCT s1.emp_id
FROM shift s1
JOIN shift s2 ON s1.emp_id = s2.emp_id
             AND s1.start_ts < s2.end_ts
             AND s1.end_ts   > s2.start_ts
WHERE s1.start_ts < s2.start_ts;
```
**Explanation**: Overlap on the same emp_id implies double-booking.

---

### 46) Merge Overlapping Intervals

**Schema**: intervals(s INT, e INT)

**Problem**: Merge overlapping [s,e] intervals into non-overlapping ranges.

**Reference (SQL)**:
```sql
WITH ord AS (
  SELECT s, e,
         SUM(CASE WHEN s > LAG(e) OVER (ORDER BY s) THEN 1 ELSE 0 END)
           OVER (ORDER BY s) AS grp
  FROM intervals
), agg AS (
  SELECT MIN(s) AS s, MAX(e) AS e, grp
  FROM (
    SELECT s,
           GREATEST(e, MAX(e) OVER (ORDER BY s ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)) AS e,
           grp
    FROM ord
  ) x
  GROUP BY grp
)
SELECT s, e FROM agg ORDER BY s;
```
**Explanation**: One SQL approach: detect new groups when current start > previous merged end, then aggregate per group.

---

### 47) Sessions that Cross Midnight

**Schema**: sessions(user_id INT, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Identify sessions whose start and end fall on different dates.

**Reference (SQL)**:
```sql
SELECT *
FROM sessions
WHERE DATE(start_ts) <> DATE(end_ts);
```
**Explanation**: Simple date extraction comparison marks crossings.

---

### 48) Find Free Time for a Single Employee

**Schema**: work(emp_id INT, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Given ordered shifts per emp_id, output the free intervals between shifts.

**Reference (SQL)**:
```sql
SELECT emp_id,
       LAG(end_ts)  OVER (PARTITION BY emp_id ORDER BY start_ts) AS free_start,
       start_ts                                             AS free_end
FROM work
QUALIFY free_start IS NOT NULL AND free_start < start_ts;
```
**Explanation**: QUALIFY (Snowflake/BigQuery) filters on window results; otherwise wrap in an outer SELECT/WHERE.

---

### 49) Longest Idle Stretch per Machine

**Schema**: tasks(machine STRING, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Compute the maximum idle duration between tasks for each machine.

**Reference (SQL)**:
```sql
WITH gaps AS (
  SELECT machine,
         start_ts - LAG(end_ts) OVER (PARTITION BY machine ORDER BY start_ts) AS idle
  FROM tasks
)
SELECT machine, MAX(idle) AS max_idle
FROM gaps
GROUP BY machine;
```
**Explanation**: Subtract previous end from current start to get idle duration.

---

### 50) Sessions Longer than 1 Hour

**Schema**: sessions(user_id INT, start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Return sessions whose duration exceeds 1 hour.

**Reference (SQL)**:
```sql
SELECT *
FROM sessions
WHERE EXTRACT(EPOCH FROM (end_ts - start_ts)) > 3600;
```
**Explanation**: Compute difference and compare in seconds; adapt for dialect.

---

### 51) Intersections with a Given Window

**Schema**: events(start_ts TIMESTAMP, end_ts TIMESTAMP)

**Problem**: Find events intersecting a given window [w_start, w_end).

**Reference (SQL)**:
```sql
SELECT *
FROM events
WHERE start_ts < :w_end
  AND end_ts   > :w_start;
```
**Explanation**: Same overlap predicate as meeting rooms; inclusive/exclusive depends on business rule.

---

### 52) Customers Above Overall Average Spend

**Schema**: orders(customer_id INT, amount NUMERIC)

**Problem**: Find customers whose average order amount is above the global average.

**Reference (SQL)**:
```sql
WITH cust AS (
  SELECT customer_id, AVG(amount) AS avg_amt
  FROM orders
  GROUP BY customer_id
), global AS (
  SELECT AVG(amount) AS gavg FROM orders
)
SELECT c.customer_id, c.avg_amt
FROM cust c CROSS JOIN global g
WHERE c.avg_amt > g.gavg;
```
**Explanation**: Compute customer averages and compare to overall average.

---

### 53) Products Never Ordered

**Schema**: products(pid INT), orders(order_id INT, pid INT)

**Problem**: List products that have never been ordered.

**Reference (SQL)**:
```sql
SELECT p.*
FROM products p
LEFT JOIN orders o ON o.pid = p.pid
WHERE o.pid IS NULL;
```
**Explanation**: Classic anti-join via LEFT JOIN + IS NULL.

---

### 54) Second Order Date per Customer

**Schema**: orders(customer_id INT, order_date DATE)

**Problem**: Return the second purchase date per customer.

**Reference (SQL)**:
```sql
SELECT customer_id, order_date
FROM (
  SELECT customer_id, order_date,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS rn
  FROM orders
) t
WHERE rn = 2;
```
**Explanation**: Use row_number within each customer ordered by date.

---

### 55) Most Recent Event per User

**Schema**: events(user_id INT, ts TIMESTAMP, payload STRING)

**Problem**: Pick the latest event row per user.

**Reference (SQL)**:
```sql
SELECT user_id, ts, payload
FROM (
  SELECT e.*, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY ts DESC) AS rn
  FROM events e
) t
WHERE rn = 1;
```
**Explanation**: Deterministic latest row via windowed row_number.

---

### 56) Customers with Orders in All Months of 2025

**Schema**: orders(customer_id INT, order_date DATE)

**Problem**: Return customers who placed at least one order in every month of 2025.

**Reference (SQL)**:
```sql
WITH m AS (
  SELECT customer_id, EXTRACT(MONTH FROM order_date) AS m
  FROM orders
  WHERE order_date >= DATE '2025-01-01' AND order_date < DATE '2026-01-01'
  GROUP BY customer_id, m
)
SELECT customer_id
FROM m
GROUP BY customer_id
HAVING COUNT(*) = 12;
```
**Explanation**: Distinct month presence per customer must be 12.

---

### 57) Users with No Activity Between Two Dates

**Schema**: events(user_id INT, dt DATE)

**Problem**: List users with no events in [2026-01-01, 2026-01-31].

**Reference (SQL)**:
```sql
SELECT u.user_id
FROM (SELECT DISTINCT user_id FROM events) u
LEFT JOIN (
  SELECT DISTINCT user_id FROM events
  WHERE dt >= DATE '2026-01-01' AND dt <= DATE '2026-01-31'
) a ON a.user_id = u.user_id
WHERE a.user_id IS NULL;
```
**Explanation**: Anti-join against the active users subquery for the target period.

---

### 58) Row-wise Maximum across Columns (UNPIVOT)

**Schema**: scores(student STRING, m1 INT, m2 INT, m3 INT)

**Problem**: Return each student and their maximum of (m1,m2,m3).

**Reference (SQL)**:
```sql
SELECT student, MAX(val) AS max_score
FROM (
  SELECT student, m1 AS val FROM scores
  UNION ALL SELECT student, m2 FROM scores
  UNION ALL SELECT student, m3 FROM scores
) x
GROUP BY student;
```
**Explanation**: Simulate UNPIVOT via UNION ALL when UNPIVOT is unavailable.

---

### 59) Top Category per Store (tie-break by name)

**Schema**: sales(store STRING, category STRING, amount NUMERIC)

**Problem**: Pick the highest-grossing category per store; break ties by category ascending.

**Reference (SQL)**:
```sql
SELECT store, category, total
FROM (
  SELECT store, category, SUM(amount) AS total,
         ROW_NUMBER() OVER (PARTITION BY store ORDER BY SUM(amount) DESC, category) AS rn
  FROM sales
  GROUP BY store, category
) t
WHERE rn = 1;
```
**Explanation**: Aggregate then rank within store; select rn=1.

---

### 60) Case-Insensitive Search

**Schema**: products(product_id INT, name STRING)

**Problem**: Find products whose name contains 'lens' (case-insensitive).

**Reference (SQL)**:
```sql
SELECT *
FROM products
WHERE LOWER(name) LIKE '%lens%';
```
**Explanation**: Lower both sides or use case-insensitive collations.

---

### 61) Extract IPv4 from Logs

**Schema**: logs(line STRING)

**Problem**: Extract the first IPv4 address from each line.

**Reference (SQL)**:
```sql
SELECT REGEXP_SUBSTR(line, '(\d{1,3}\.){3}\d{1,3}') AS ip
FROM logs;
```
**Explanation**: Use REGEXP_SUBSTR; escape backslashes appropriately for the engine.

---

### 62) Normalize Names for Joining

**Schema**: src(name STRING), dim(name STRING)

**Problem**: Join on normalized names: trim, lower, replace spaces with underscore.

**Reference (SQL)**:
```sql
SELECT *
FROM (
  SELECT LOWER(REPLACE(TRIM(name), ' ', '_')) AS k FROM src
) s
JOIN (
  SELECT LOWER(REPLACE(TRIM(name), ' ', '_')) AS k FROM dim
) d ON d.k = s.k;
```
**Explanation**: Normalize both sides in the same way to improve match rate.

---

### 63) Mask Emails to Username@****

**Schema**: users(email STRING)

**Problem**: Return masked emails showing only the user part and domain TLD.

**Reference (SQL)**:
```sql
SELECT CONCAT(SPLIT_PART(email,'@',1), '@', '****.', SPLIT_PART(SPLIT_PART(email,'@',2), '.', -1)) AS masked
FROM users;
```
**Explanation**: Split at @ and keep TLD; mask the domain label.

---

### 64) Count Words in a Free-Text Column

**Schema**: docs(id INT, body STRING)

**Problem**: Approximate word count per row, splitting on spaces.

**Reference (SQL)**:
```sql
SELECT id,
       CARDINALITY(SPLIT(REGEXP_REPLACE(TRIM(body), '
', ' '), ' ')) AS word_count
FROM docs;
```
**Explanation**: Simple split-based count; refine with regex for punctuation as needed.

---

### 65) Find Palindromic Strings

**Schema**: s(txt STRING)

**Problem**: Return rows where txt is a palindrome (case-insensitive).

**Reference (SQL)**:
```sql
SELECT *
FROM s
WHERE LOWER(txt) = REVERSE(LOWER(txt));
```
**Explanation**: Compare normalized string to its reverse.

---

### 66) Emails by Domain

**Schema**: users(email STRING)

**Problem**: Aggregate counts by email domain.

**Reference (SQL)**:
```sql
SELECT SPLIT_PART(LOWER(email),'@',2) AS domain, COUNT(*) AS cnt
FROM users
GROUP BY domain
ORDER BY cnt DESC;
```
**Explanation**: Split on @ to get domain and aggregate.

---

### 67) Left/Right/Trim Examples

**Schema**: names(full STRING)

**Problem**: Return first 5 chars (LEFT), last 3 chars (RIGHT), and trimmed string.

**Reference (SQL)**:
```sql
SELECT LEFT(full, 5)  AS first5,
       RIGHT(full, 3) AS last3,
       TRIM(full)     AS trimmed
FROM names;
```
**Explanation**: Demonstrates common text functions.

---

### 68) Extract JSON Field

**Schema**: events(payload JSON)

**Problem**: Get user.id from JSON payload.

**Reference (SQL)**:
```sql
SELECT JSON_VALUE(payload, '$.user.id') AS user_id
FROM events;
```
**Explanation**: Use JSON_VALUE/EXTRACT depending on dialect.

---

### 69) Flatten JSON Array

**Schema**: events(payload JSON)

**Problem**: Explode items[] from payload with one row per item.

**Reference (SQL)**:
```sql
SELECT e.*, j.value AS item
FROM events e,
LATERAL JSON_TABLE(e.payload, '$.items[*]' COLUMNS(value VARCHAR(200) PATH '$')) j;
```
**Explanation**: JSON_TABLE (or UNNEST/EXPLODE) expands arrays to rows.

---

### 70) Build JSON Array per Order

**Schema**: order_items(order_id INT, item STRING)

**Problem**: Aggregate items of each order into a JSON array.

**Reference (SQL)**:
```sql
SELECT order_id, JSON_ARRAYAGG(item) AS items
FROM order_items
GROUP BY order_id;
```
**Explanation**: JSON array aggregation serializes grouped rows.

---

### 71) Array Length and First Element

**Schema**: arr_tbl(id INT, tags ARRAY<STRING>)

**Problem**: Return array size and first tag for each row.

**Reference (SQL)**:
```sql
SELECT id, CARDINALITY(tags) AS array_size, tags[1] AS first_tag
FROM arr_tbl;
```
**Explanation**: Array subscripts and cardinality vary by engine; many are 1-based.

---

### 72) Filter Rows by JSON Predicate

**Schema**: events(payload JSON)

**Problem**: Select rows where payload.status == 'ok' and payload.count > 10.

**Reference (SQL)**:
```sql
SELECT *
FROM events
WHERE JSON_VALUE(payload, '$.status') = 'ok'
  AND CAST(JSON_VALUE(payload, '$.count') AS INT) > 10;
```
**Explanation**: Compare scalar JSON values after casting to the proper type.

---

### 73) Pivot JSON Keys to Columns (simple)

**Schema**: kv(payload JSON)

**Problem**: Extract payload{"a","b","c"} into separate columns.

**Reference (SQL)**:
```sql
SELECT JSON_VALUE(payload,'$.a') AS a,
       JSON_VALUE(payload,'$.b') AS b,
       JSON_VALUE(payload,'$.c') AS c
FROM kv;
```
**Explanation**: Direct projection of well-known keys.

---

### 74) Aggregate JSON per Group

**Schema**: x(g STRING, k STRING, v STRING)

**Problem**: Build a map-like JSON object of (k→v) per group g (dialect permitting).

**Reference (SQL)**:
```sql
SELECT g, JSON_OBJECTAGG(k VALUE v) AS obj
FROM x
GROUP BY g;
```
**Explanation**: JSON_OBJECTAGG / MAP_AGG depends on engine; provides key→value object.

---

### 75) Search Array for Value

**Schema**: arr_tbl(id INT, tags ARRAY<STRING>)

**Problem**: Return rows where 'critical' is present in tags array.

**Reference (SQL)**:
```sql
SELECT *
FROM arr_tbl
WHERE ARRAY_CONTAINS(tags, 'critical');
```
**Explanation**: ARRAY_CONTAINS/ANY_MATCH semantics vary by engine.

---

### 76) Calendar Table (Recursive CTE)

**Problem**: Generate dates from 2026-01-01 to 2026-12-31.

**Reference (SQL)**:
```sql
WITH RECURSIVE cal(d) AS (
  SELECT DATE '2026-01-01'
  UNION ALL
  SELECT d + INTERVAL '1' DAY FROM cal WHERE d < DATE '2026-12-31'
)
SELECT * FROM cal;
```
**Explanation**: Use generate_series() where supported; recursive CTE is ANSI-friendly.

---

### 77) Month Start/End from Date

**Schema**: orders(order_date DATE)

**Problem**: Return month_start and month_end for each order_date.

**Reference (SQL)**:
```sql
SELECT order_date,
       DATE_TRUNC('month', order_date) AS month_start,
       (DATE_TRUNC('month', order_date) + INTERVAL '1' MONTH - INTERVAL '1' DAY) AS month_end
FROM orders;
```
**Explanation**: DATE_TRUNC to floor; add a month then subtract a day for month end.

---

### 78) ISO Week Number

**Schema**: events(dt DATE)

**Problem**: Return ISO week of year for each date.

**Reference (SQL)**:
```sql
SELECT EXTRACT(ISOWEEK FROM dt) AS iso_week
FROM events;
```
**Explanation**: Use ISO week to handle year edges consistently.

---

### 79) Monthly Cohort of Signups

**Schema**: users(user_id INT, signup_date DATE)

**Problem**: Assign cohort_month = first day of signup month.

**Reference (SQL)**:
```sql
SELECT user_id, DATE_TRUNC('month', signup_date) AS cohort_month
FROM users;
```
**Explanation**: Cohort bucketing via DATE_TRUNC.

---

### 80) Retention: Users Active in Both Month M and M+1

**Schema**: events(user_id INT, dt DATE)

**Problem**: List users active in consecutive months.

**Reference (SQL)**:
```sql
WITH m AS (
  SELECT DISTINCT user_id, DATE_TRUNC('month', dt) AS m
  FROM events
)
SELECT a.user_id, a.m AS month
FROM m a
JOIN m b ON a.user_id = b.user_id AND b.m = a.m + INTERVAL '1' MONTH;
```
**Explanation**: Self-join month buckets offset by one month to capture consecutive activity.

---

### 81) Last 30 Days (half-open)

**Schema**: log(ts TIMESTAMP)

**Problem**: Select rows in the last 30 whole days from current_date.

**Reference (SQL)**:
```sql
SELECT *
FROM log
WHERE ts >= CURRENT_DATE - INTERVAL '30' DAY
  AND ts <  CURRENT_DATE + INTERVAL '1' DAY;
```
**Explanation**: Half-open boundary helps avoid fencepost errors.

---

### 82) End-of-Month Snapshot Balance

**Schema**: balances(account_id INT, dt DATE, balance NUMERIC)

**Problem**: Pick the last known balance per account per month.

**Reference (SQL)**:
```sql
SELECT account_id, month, balance
FROM (
  SELECT account_id,
         DATE_TRUNC('month', dt) AS month,
         balance,
         ROW_NUMBER() OVER (PARTITION BY account_id, DATE_TRUNC('month', dt) ORDER BY dt DESC) AS rn
  FROM balances
) t
WHERE rn = 1;
```
**Explanation**: Row-number on (account,month) selects latest record in month.

---

### 83) Weekday vs Weekend Split

**Schema**: events(dt DATE)

**Problem**: Aggregate counts for weekday vs weekend.

**Reference (SQL)**:
```sql
SELECT CASE WHEN EXTRACT(DOW FROM dt) IN (6,0) THEN 'weekend' ELSE 'weekday' END AS day_type,
       COUNT(*) AS cnt
FROM events
GROUP BY 1;
```
**Explanation**: DOW 0/6 often map to Sunday/Saturday (dialect-specific).

---

### 84) Employees Earning More Than Their Managers

**Schema**: employees(emp_id INT, name STRING, manager_id INT, salary INT)

**Problem**: List employees whose salary exceeds their manager’s.

**Reference (SQL)**:
```sql
SELECT e.emp_id, e.name
FROM employees e
JOIN employees m ON m.emp_id = e.manager_id
WHERE e.salary > m.salary;
```
**Explanation**: Self-join employees to managers and compare salaries.

---

### 85) Find the Median of Two Columns (stack then median)

**Schema**: vals(a INT, b INT)

**Problem**: Compute the median across two numeric columns per row group.

**Reference (SQL)**:
```sql
WITH stacked AS (
  SELECT a AS v FROM vals
  UNION ALL
  SELECT b FROM vals
)
SELECT PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY v) AS median
FROM stacked;
```
**Explanation**: Stack columns into rows, then apply ordered-set percentile.

---

### 86) K Most Frequent Words (free text)

**Schema**: docs(id INT, body STRING)

**Problem**: Tokenize by spaces and return top 10 words.

**Reference (SQL)**:
```sql
WITH tokens AS (
  SELECT LOWER(t) AS w
  FROM docs,
       LATERAL SPLIT(TRIM(REGEXP_REPLACE(body,'
',' ')),' ') AS t
)
SELECT w, COUNT(*) AS cnt
FROM tokens
GROUP BY w
ORDER BY cnt DESC
FETCH FIRST 10 ROWS ONLY;
```
**Explanation**: Approximate tokenization by splitting on spaces; refine regex as needed.

---

### 87) Two Largest per Group without Window Functions

**Schema**: numbers(g STRING, v INT)

**Problem**: Return two largest values per group using self-join.

**Reference (SQL)**:
```sql
SELECT n1.g, n1.v
FROM numbers n1
LEFT JOIN numbers n2 ON n2.g = n1.g AND n2.v > n1.v
GROUP BY n1.g, n1.v
HAVING COUNT(n2.v) < 2;
```
**Explanation**: Count how many values are greater; keep those with fewer than 2 greater values.

---

### 88) Rows Where a Value Changes (Changepoints)

**Schema**: signals(ts TIMESTAMP, state STRING)

**Problem**: Return timestamps where state differs from the previous row.

**Reference (SQL)**:
```sql
SELECT *
FROM (
  SELECT s.*,
         LAG(state) OVER (ORDER BY ts) AS prev_state
  FROM signals s
) t
WHERE prev_state IS NULL OR state <> prev_state;
```
**Explanation**: Detect changepoints via LAG comparison.

---

### 89) Pivot Months to Columns (conditional aggregation)

**Schema**: sales(dt DATE, amount NUMERIC)

**Problem**: Show totals by month as columns (Jan..Dec).

**Reference (SQL)**:
```sql
SELECT SUM(CASE WHEN EXTRACT(MONTH FROM dt)=1  THEN amount ELSE 0 END) AS jan,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=2  THEN amount ELSE 0 END) AS feb,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=3  THEN amount ELSE 0 END) AS mar,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=4  THEN amount ELSE 0 END) AS apr,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=5  THEN amount ELSE 0 END) AS may,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=6  THEN amount ELSE 0 END) AS jun,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=7  THEN amount ELSE 0 END) AS jul,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=8  THEN amount ELSE 0 END) AS aug,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=9  THEN amount ELSE 0 END) AS sep,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=10 THEN amount ELSE 0 END) AS oct,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=11 THEN amount ELSE 0 END) AS nov,
       SUM(CASE WHEN EXTRACT(MONTH FROM dt)=12 THEN amount ELSE 0 END) AS "dec"
FROM sales;
```
**Explanation**: Manual pivot using CASE expressions; portable across engines.

---

### 90) Symmetric Difference of Two Sets

**Schema**: a(x INT), b(x INT)

**Problem**: Rows in A or B but not in both (A△B).

**Reference (SQL)**:
```sql
(SELECT x FROM a EXCEPT SELECT x FROM b)
UNION ALL
(SELECT x FROM b EXCEPT SELECT x FROM a);
```
**Explanation**: EXCEPT removes common rows; UNION ALL combines the two differences.

---

### 91) Top Customers by Monthly Growth

**Schema**: orders(customer_id INT, order_date DATE, amount NUMERIC)

**Problem**: For each month, list the customer with maximum MoM growth.

**Reference (SQL)**:
```sql
WITH m AS (
  SELECT customer_id,
         DATE_TRUNC('month', order_date) AS month,
         SUM(amount) AS amt
  FROM orders
  GROUP BY 1,2
), g AS (
  SELECT customer_id, month, amt,
         amt - LAG(amt) OVER (PARTITION BY customer_id ORDER BY month) AS mom
  FROM m
)
SELECT month, customer_id, mom
FROM (
  SELECT g.*, ROW_NUMBER() OVER (PARTITION BY month ORDER BY mom DESC NULLS LAST) AS rn
  FROM g
) t
WHERE rn = 1;
```
**Explanation**: Compute per-customer MoM, then pick the top per month.

---

### 92) Find Rows with Duplicate Primary-Key Candidates

**Schema**: users(email STRING, created_at TIMESTAMP)

**Problem**: Emails that occur multiple times with earliest creation date.

**Reference (SQL)**:
```sql
SELECT email, MIN(created_at) AS first_seen, COUNT(*) AS cnt
FROM users
GROUP BY email
HAVING COUNT(*) > 1;
```
**Explanation**: Duplicates by natural key; report earliest occurrence.

---

### 93) First Value per Partition without Window Functions

**Schema**: events(user_id INT, ts TIMESTAMP, payload STRING)

**Problem**: Earliest event per user using GROUP BY + MIN and a join.

**Reference (SQL)**:
```sql
WITH first_ts AS (
  SELECT user_id, MIN(ts) AS mts
  FROM events
  GROUP BY user_id
)
SELECT e.*
FROM events e
JOIN first_ts f ON f.user_id = e.user_id AND f.mts = e.ts;
```
**Explanation**: Two-step: compute MIN timestamp per user, then join back to get the row.

---

### 94) Cumulative Distinct Users (approx)

**Schema**: events(user_id INT, dt DATE)

**Problem**: Approx distinct users seen up to each month.

**Reference (SQL)**:
```sql
WITH m AS (
  SELECT DATE_TRUNC('month', dt) AS month, user_id
  FROM events
  GROUP BY 1, user_id
)
SELECT month,
       COUNT(DISTINCT user_id) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_users
FROM (
  SELECT DISTINCT month FROM m
) months
LEFT JOIN m USING (month)
ORDER BY month;
```
**Explanation**: Distinct-by-month then cumulative distinct over months (some engines need workarounds for window DISTINCT).

---

### 95) Null-Safe Equality Emulation

**Schema**: a(id INT, v STRING), b(id INT, v STRING)

**Problem**: Join treating NULL=NULL as equal.

**Reference (SQL)**:
```sql
SELECT *
FROM a
JOIN b ON a.id = b.id AND (a.v = b.v OR (a.v IS NULL AND b.v IS NULL));
```
**Explanation**: ANSI SQL lacks <=>; emulate via explicit NULL checks.

---

### 96) Histogram Bucket Counts

**Schema**: metrics(x DOUBLE)

**Problem**: Count values into buckets [0,10),[10,20),[20,∞).

**Reference (SQL)**:
```sql
SELECT CASE WHEN x < 10 THEN '[0,10)'
            WHEN x < 20 THEN '[10,20)'
            ELSE '[20,∞)'
       END AS bucket,
       COUNT(*) AS cnt
FROM metrics
GROUP BY bucket
ORDER BY bucket;
```
**Explanation**: Bucketize via CASE and aggregate.

---

### 97) Find Median per Group (approx)

**Schema**: orders(store STRING, amount DOUBLE)

**Problem**: Approximate median spend per store.

**Reference (SQL)**:
```sql
SELECT store, APPROX_PERCENTILE(amount, 0.5) AS median
FROM orders
GROUP BY store;
```
**Explanation**: Approximate percentile is scalable for large data.

---

### 98) List Skipped Sequence Numbers

**Schema**: seq(n INT)

**Problem**: Given integer sequence table, find missing integers between min and max.

**Reference (SQL)**:
```sql
WITH bounds AS (
  SELECT MIN(n) AS mn, MAX(n) AS mx FROM seq
), cal AS (
  SELECT mn AS n FROM bounds
  UNION ALL
  SELECT n+1 FROM cal, bounds WHERE n < mx
)
SELECT cal.n AS missing
FROM cal
LEFT JOIN seq s ON s.n = cal.n
WHERE s.n IS NULL;
```
**Explanation**: Recursive generator compared with existing rows to find holes.

---

### 99) Round Timestamps to Nearest 5 Minutes

**Schema**: events(ts TIMESTAMP)

**Problem**: Return timestamp rounded to nearest 5-minute boundary.

**Reference (SQL)**:
```sql
SELECT DATEADD(minute, 5 * ROUND(EXTRACT(EPOCH FROM ts)/60.0/5), TIMESTAMP '1970-01-01') AS ts_5
FROM events;
```
**Explanation**: Compute epoch minutes, divide by 5, round, then reconstruct; adjust functions per dialect.

---

### 100) Find K-th Order per Customer (offset approach)

**Schema**: orders(customer_id INT, order_date DATE, order_id INT)

**Problem**: Return the 3rd order per customer (if any).

**Reference (SQL)**:
```sql
SELECT customer_id, order_id, order_date
FROM (
  SELECT customer_id, order_id, order_date,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS rn
  FROM orders
) t
WHERE rn = 3;
```
**Explanation**: Windowing orders by date and filtering rn = K.

---

